# Intermediate: Model Evaluation and Metrics

Learn to properly evaluate your models beyond simple accuracy!

Topics covered:
- Classification metrics (Precision, Recall, F1)
- Confusion matrices
- ROC curves and AUC
- Multi-class classification metrics
- Regression metrics
- Cross-validation
- Model interpretation techniques

## Why Proper Evaluation Matters

```
Scenario: Medical Diagnosis (99% healthy, 1% sick)

Model A: Always predicts "healthy"
  Accuracy: 99% ✓
  But misses ALL sick patients! ✗

Model B: Balanced classifier
  Accuracy: 95%
  But catches 90% of sick patients ✓

Which is better? Accuracy alone doesn't tell the story!
```


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    f1_score, accuracy_score, precision_score, recall_score
)
from sklearn.model_selection import KFold
import pandas as pd

print("Evaluation tools loaded!")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Classification Metrics

Understanding the metrics:

```
Confusion Matrix:
                 Predicted
               Neg    Pos
Actual  Neg    TN     FP
        Pos    FN     TP

TN = True Negative  (correctly predicted negative)
FP = False Positive (incorrectly predicted positive) - Type I error
FN = False Negative (incorrectly predicted negative) - Type II error  
TP = True Positive  (correctly predicted positive)

Metrics:
• Accuracy = (TP + TN) / (TP + TN + FP + FN)
• Precision = TP / (TP + FP)  - "Of all positive predictions, how many were correct?"
• Recall = TP / (TP + FN)     - "Of all actual positives, how many did we find?"
• F1-Score = 2 × (Precision × Recall) / (Precision + Recall)
```


In [ ]:
# Example: Binary classification results
y_true = np.array([0, 1, 1, 0, 1, 1, 0, 0, 1, 0])
y_pred = np.array([0, 1, 1, 0, 0, 1, 0, 1, 1, 0])

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Binary Classification Metrics:")
print("="*50)
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")
print()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)
print()
print(f"True Negatives:  {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives:  {cm[1,1]}")


## 2. Visualizing Confusion Matrix


In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, normalize=False, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
    else:
        fmt = 'd'
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Count' if not normalize else 'Proportion'})
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    return cm

# Example with CIFAR-10 classes
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Simulate predictions
np.random.seed(42)
n_samples = 1000
y_true_multi = np.random.randint(0, 10, n_samples)
y_pred_multi = y_true_multi.copy()
# Add some errors
errors = np.random.choice(n_samples, size=200, replace=False)
y_pred_multi[errors] = np.random.randint(0, 10, 200)

cm = plot_confusion_matrix(y_true_multi, y_pred_multi, classes, normalize=True,
                           title='Normalized Confusion Matrix - CIFAR-10')
plt.show()

print("Confusion matrix plotted!")


## 3. Multi-Class Classification Metrics


In [ ]:
# Detailed classification report
print("Classification Report:")
print("="*70)
report = classification_report(y_true_multi, y_pred_multi, target_names=classes)
print(report)
print()

# Per-class metrics
precision_per_class = precision_score(y_true_multi, y_pred_multi, average=None)
recall_per_class = recall_score(y_true_multi, y_pred_multi, average=None)
f1_per_class = f1_score(y_true_multi, y_pred_multi, average=None)

# Create DataFrame for better visualization
metrics_df = pd.DataFrame({
    'Class': classes,
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class
})

print("Per-Class Metrics:")
print(metrics_df.to_string(index=False))
print()

# Different averaging strategies
print("Averaging Strategies:")
print(f"Macro avg (unweighted mean):    {f1_score(y_true_multi, y_pred_multi, average='macro'):.3f}")
print(f"Weighted avg (weighted by support): {f1_score(y_true_multi, y_pred_multi, average='weighted'):.3f}")
print(f"Micro avg (global):             {f1_score(y_true_multi, y_pred_multi, average='micro'):.3f}")


## 4. ROC Curves and AUC

ROC (Receiver Operating Characteristic) curves visualize trade-off between True Positive Rate and False Positive Rate.

```
ROC Curve:
TPR
 ^
 |    Perfect
 |    Classifier
 |     ╱|
 |    ╱ |
 |   ╱  |  Good
 |  ╱   | Classifier
 | ╱    |╱
 |╱_____|_____ Random
 |      ╱     Classifier
 └─────────────> FPR

AUC = Area Under Curve
• AUC = 1.0: Perfect
• AUC = 0.9-1.0: Excellent
• AUC = 0.8-0.9: Good
• AUC = 0.7-0.8: Fair
• AUC = 0.5: Random
```


In [ ]:
# Generate probability scores for binary classification
np.random.seed(42)
y_true_binary = np.random.randint(0, 2, 1000)
y_scores = np.random.rand(1000)
# Make scores somewhat correlated with true labels
y_scores[y_true_binary == 1] += 0.3
y_scores = np.clip(y_scores, 0, 1)

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {roc_auc:.3f}")
print()

# Find optimal threshold
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold: {optimal_threshold:.3f}")
print(f"TPR at optimal: {tpr[optimal_idx]:.3f}")
print(f"FPR at optimal: {fpr[optimal_idx]:.3f}")


## 5. Precision-Recall Curve

Better than ROC for imbalanced datasets.

```
When to use which:
• ROC Curve: Balanced datasets
• PR Curve: Imbalanced datasets (e.g., rare disease detection)

Precision-Recall Trade-off:
High Precision → Few false positives (conservative)
High Recall → Few false negatives (aggressive)
```


In [ ]:
# Compute Precision-Recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_true_binary, y_scores)
pr_auc = auc(recall, precision)

# Plot Precision-Recall curve
plt.figure(figsize=(10, 8))
plt.plot(recall, precision, color='blue', lw=2,
         label=f'PR curve (AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.show()

print(f"Precision-Recall AUC: {pr_auc:.3f}")
print()

# F1 scores at different thresholds
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_f1_idx = np.argmax(f1_scores[:-1])  # Last point has issues
best_threshold = pr_thresholds[best_f1_idx]
print(f"Best F1 threshold: {best_threshold:.3f}")
print(f"Best F1 score: {f1_scores[best_f1_idx]:.3f}")
print(f"Precision at best F1: {precision[best_f1_idx]:.3f}")
print(f"Recall at best F1: {recall[best_f1_idx]:.3f}")


## 6. Regression Metrics


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Generate regression data
np.random.seed(42)
y_true_reg = np.random.randn(100) * 10 + 50
y_pred_reg = y_true_reg + np.random.randn(100) * 3

# Calculate regression metrics
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true_reg, y_pred_reg)
r2 = r2_score(y_true_reg, y_pred_reg)

# Mean Absolute Percentage Error
mape = np.mean(np.abs((y_true_reg - y_pred_reg) / y_true_reg)) * 100

print("Regression Metrics:")
print("="*50)
print(f"Mean Squared Error (MSE):  {mse:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"Mean Absolute Error (MAE): {mae:.3f}")
print(f"R² Score:                  {r2:.3f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print()

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(y_true_reg, y_pred_reg, alpha=0.5)
plt.plot([y_true_reg.min(), y_true_reg.max()], 
         [y_true_reg.min(), y_true_reg.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('Regression: True vs Predicted Values')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Residual plot
residuals = y_true_reg - y_pred_reg
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_reg, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)
plt.show()


## 7. Cross-Validation with PyTorch


In [ ]:
# K-Fold Cross-Validation
from torch.utils.data import SubsetRandomSampler

def k_fold_cross_validation(model_class, dataset, k=5, epochs=10):
    # Initialize K-Fold
    kfold = KFold(n_splits=k, shuffle=True, random_state=42)
    
    results = []
    
    for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
        print(f"\nFold {fold + 1}/{k}")
        print("-" * 50)
        
        # Create data samplers
        train_subsampler = SubsetRandomSampler(train_ids)
        val_subsampler = SubsetRandomSampler(val_ids)
        
        # Create data loaders
        train_loader = torch.utils.data.DataLoader(
            dataset, batch_size=64, sampler=train_subsampler
        )
        val_loader = torch.utils.data.DataLoader(
            dataset, batch_size=64, sampler=val_subsampler
        )
        
        # Initialize model
        model = model_class().to(device)
        optimizer = torch.optim.Adam(model.parameters())
        criterion = nn.CrossEntropyLoss()
        
        # Train
        for epoch in range(epochs):
            model.train()
            for data, target in train_loader:
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
        
        # Validate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()
        
        accuracy = 100. * correct / total
        results.append(accuracy)
        print(f"Validation Accuracy: {accuracy:.2f}%")
    
    print("\n" + "="*50)
    print(f"Average Accuracy: {np.mean(results):.2f}% (+/- {np.std(results):.2f}%)")
    return results

print("K-Fold Cross-Validation function defined!")
print("\nUsage:")
print("  results = k_fold_cross_validation(MyModel, dataset, k=5)")


## 8. Model Interpretation Techniques


In [ ]:
# Feature importance visualization
def plot_feature_importance(model, feature_names, top_n=20):
    # For linear models, weights indicate importance
    if hasattr(model, 'fc') and hasattr(model.fc, 'weight'):
        weights = model.fc.weight.data.cpu().numpy()
        
        if len(weights.shape) == 2 and weights.shape[0] == 1:
            # Binary classification
            importance = np.abs(weights[0])
        else:
            # Multi-class: average importance across classes
            importance = np.abs(weights).mean(axis=0)
        
        # Get top features
        top_indices = np.argsort(importance)[-top_n:]
        top_importance = importance[top_indices]
        top_names = [feature_names[i] if i < len(feature_names) else f"Feature {i}" 
                     for i in top_indices]
        
        # Plot
        plt.figure(figsize=(10, 8))
        plt.barh(range(len(top_importance)), top_importance)
        plt.yticks(range(len(top_importance)), top_names)
        plt.xlabel('Absolute Weight (Importance)')
        plt.title(f'Top {top_n} Feature Importances')
        plt.tight_layout()
        plt.show()

print("Feature importance visualization function defined!")
print()

# Grad-CAM for CNNs (Class Activation Mapping)
print("For CNN visualization:")
print("  • Use Grad-CAM to see which regions the model focuses on")
print("  • Visualize filters to understand what features are learned")
print("  • Plot activation maps at different layers")
print()

print("Advanced interpretation tools:")
print("  • LIME (Local Interpretable Model-agnostic Explanations)")
print("  • SHAP (SHapley Additive exPlanations)")
print("  • Integrated Gradients")
print("  • Attention visualization (for Transformers)")


## 9. Evaluation Best Practices


In [ ]:
print("Model Evaluation Best Practices:")
print("="*70)
print()

print("1. ALWAYS USE A HELD-OUT TEST SET")
print("   ✓ Train/Val/Test split: 70%/15%/15% or 60%/20%/20%")
print("   ✓ Never touch test set until final evaluation")
print("   ✗ Don't tune hyperparameters on test set")
print()

print("2. CHOOSE APPROPRIATE METRICS")
print("   • Balanced classification: Accuracy, F1")
print("   • Imbalanced classification: Precision, Recall, PR-AUC")
print("   • Multi-class: Macro/Weighted F1, Confusion Matrix")
print("   • Regression: RMSE, MAE, R²")
print()

print("3. LOOK BEYOND SINGLE METRICS")
print("   ✓ Plot confusion matrix")
print("   ✓ Examine per-class performance")
print("   ✓ Analyze failure cases")
print("   ✓ Check for class imbalance effects")
print()

print("4. USE CROSS-VALIDATION")
print("   ✓ For small datasets (<10K samples)")
print("   ✓ To get confidence intervals")
print("   ✓ For robust hyperparameter tuning")
print()

print("5. STATISTICAL SIGNIFICANCE")
print("   ✓ Run multiple training runs with different seeds")
print("   ✓ Report mean and standard deviation")
print("   ✓ Use statistical tests when comparing models")
print()

print("6. REAL-WORLD CONSIDERATIONS")
print("   • Consider inference time")
print("   • Check model size (memory)")
print("   • Evaluate on edge cases")
print("   • Test for robustness to noise")
print("   • Check for biases")


## 📝 Summary

✅ Calculated classification metrics (Precision, Recall, F1)  
✅ Created and visualized confusion matrices  
✅ Plotted ROC and PR curves  
✅ Computed regression metrics  
✅ Implemented cross-validation  
✅ Learned model interpretation techniques  
✅ Applied evaluation best practices  

### Key Metrics Cheat Sheet

**Classification:**
- Accuracy: Overall correctness
- Precision: Positive prediction accuracy
- Recall: Ability to find all positives
- F1: Balance of precision and recall
- ROC-AUC: Overall discrimination ability
- PR-AUC: Performance on imbalanced data

**Regression:**
- MSE/RMSE: Average squared error
- MAE: Average absolute error
- R²: Proportion of variance explained
- MAPE: Percentage error

### When to Use What

| Scenario | Best Metric |
|----------|-------------|
| Balanced classes | Accuracy, F1 |
| Imbalanced classes | Precision/Recall, PR-AUC |
| Cost-sensitive | Custom metric with weights |
| Multi-class | Macro/Weighted F1 |
| Ranking | ROC-AUC |
| Regression | RMSE, R² |

### Next Steps

- **Next**: `06_data_advanced.ipynb` - Advanced data processing
- **Practice**: Evaluate your models with multiple metrics
- **Challenge**: Implement stratified k-fold cross-validation


## 🎯 Practice Exercises


In [ ]:
# Exercise 1: Implement a function to find the best threshold for F1
# Your code here:


# Exercise 2: Create a multi-class ROC curve (one-vs-rest)
# Your code here:


# Exercise 3: Implement stratified k-fold cross-validation
# Your code here:


# Exercise 4: Calculate confidence intervals for your metrics
# Your code here:


# Exercise 5: Create a comprehensive evaluation report function
# Your code here:
